In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import shutil
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

In [ ]:
DATA_ROOT = "/data"
N_BANKS = 3
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)

In [ ]:
TRANSACTION_PATH=(
    "./data/"
    "train_transaction.csv"
)

IDENTITY_PATH=(
    "./data/"
    "train_identity.csv"
)

In [ ]:
train_transaction = pd.read_csv(TRANSACTION_PATH)
train_identity = pd.read_csv(IDENTITY_PATH)

In [ ]:
print("shape transaction : ", train_transaction.shape)
print("shape identity : ", train_identity.shape)
print(" duplicated TransactionID in identity :",
      train_identity["TransactionID"].duplicated().sum())

shape transaction :  (590540, 394)
shape identity :  (144233, 41)
 duplicated TransactionID in identity : 0


In [ ]:
data = train_transaction.merge(train_identity, on = "TransactionID", how="left", validate="one_to_one")

print("merged shape : ", data.shape)
print(" Fraud rate : ", data["isFraud"].mean())

merged shape :  (590540, 434)
 Fraud rate :  0.03499000914417313


In [ ]:
TARGET = "isFraud"

X = data.drop(columns=[TARGET, "TransactionID"])
y = data[TARGET]

In [ ]:
# l'ordre est toujours Xtrain, Xtest ou val, ytrain, ytest ou val 
# le random state est pour avoir une graine de mélange pour retomber sur le meme dataset si on relance
# stratify c'est pour avoir la même proportion de valeur cible dans train et validaiton
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_val.mean())

X_train: (472432, 432)
X_val: (118108, 432)
Train fraud rate: 0.03498916246147594
Validation fraud rate: 0.0349933958749619


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# le but de OrdinalEncoder est d'encoder les variables qui ne sont pas number (comme string ou object). 
# Nos modèles de ML ne comprénnes uniquements les variables numériques

X_train_enc = X_train.copy()
X_val_enc = X_val.copy()

cat_cols_string = X_train.select_dtypes(
    include=["object", "category","string"]
).columns.tolist()


In [ ]:
# indispensable, avant d'encoder il ne faut pas de null et surtout que la colonne 
# soit d'un type unique (ici string comme ça on passe de string à numerical)
for col in cat_cols_string:
    X_train_enc[col] = (
        X_train_enc[col].astype("string").fillna("__MISSING__")
    )
    X_val_enc[col] = (    
        X_val_enc[col].astype("string").fillna("__MISSING__")
    )

In [ ]:
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [ ]:
if cat_cols_string:
    X_train_enc[cat_cols_string] = encoder.fit_transform(
        X_train_enc[cat_cols_string]
    )
    X_val_enc[cat_cols_string] = encoder.fit_transform(
            X_val_enc[cat_cols_string]
    )
    

In [ ]:
X_train_enc = X_train_enc.replace(
    [np.inf, -np.inf], np.nan
)

X_val_enc = X_val_enc.replace(
    [np.inf, -np.inf], np.nan
)

In [ ]:
X_val_enc = X_val_enc[X_train_enc.columns]

print("Encoded train:", X_train_enc.shape)
print("Encoded validation:", X_val_enc.shape)
print("Categorical features:", len(cat_cols_string))

Encoded train: (472432, 432)
Encoded validation: (118108, 432)
Categorical features: 31


In [ ]:
def split_into_banks(
    X_train,
    y_train,
    X_val,
    y_val,
    n_banks,
    output_dir="/kaggle/working/horizontal_xgb_data",
    seed=42
):

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)


    for i in range(1, 21):
        old_dir = output_dir/ f"site-{i}"
        if old_dir.exists():
            shutil.rmtree(old_dir)
    X_array = np.asarray(X_train, dtype=np.float32)
    y_array = np.asarray(y_train).reshape(-1).astype(np.int32)

    Xv = np.asarray(X_val, dtype=np.float32)
    yv = np.asarray(y_val).reshape(-1).astype(np.int32)

    splitter = StratifiedKFold(
        n_splits=n_banks,
        shuffle=True,
        random_state = seed
    )

    summary = []

    for bank_id, (_,idx) in enumerate(
        splitter.split(X_array, y_array), start=1
    ):
        bank_dir = output_dir /f"site-{bank_id}"
        bank_dir.mkdir(parents=True, exist_ok=True)

        train_df = pd.DataFrame(
            np.column_stack([y_array[idx], X_array[idx]])
        )
        train_df.to_csv(
            bank_dir / "train.csv",
            index= False,
            header=False
        )
        valid_df = pd.DataFrame(
            np.column_stack([yv, Xv])
        )
        valid_df.to_csv(
            bank_dir / "valid.csv",
            index=False,
            header=False
        )

        summary.append({
            "bank": f"site-{bank_id}",
            "transactions": len(idx),
            "fraud": int(y_array[idx].sum()),
            "fraud_rate_%": round(
                float(y_array[idx].mean() * 100), 3
            ),
            "train_exists": (bank_dir / "train.csv").exists(),
            "valid_exists": (bank_dir / "valid.csv").exists()
        })

    summary_df = pd.DataFrame(summary)
    print(summary_df.to_string(index=False))
    return summary_df

summary = split_into_banks(
    X_train_enc,
    y_train,
    X_val_enc,
    y_val,
    n_banks=N_BANKS,
    output_dir=DATA_ROOT
)


  bank  transactions  fraud  fraud_rate_%  train_exists  valid_exists
site-1        157478   5510         3.499          True          True
site-2        157477   5510         3.499          True          True
site-3        157477   5510         3.499          True          True
